# Pemisahan Pada Dataset Final untuk Trainning Validation & Testing

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import re

print("=" * 75)
print("🚀 FASE 2: PEMBERSIHAN MUTLAK & STRATIFIED SPLIT 70/15/15")
print("=" * 75)

# 1. BACA FILE MASTER
input_csv = "../data/raw/dataset_politik_balanced.csv"
if not os.path.exists(input_csv):
    input_csv = "data/raw/dataset_politik_balanced.csv"

df_master = pd.read_csv(input_csv)
print(f"Total Master Dataset Awal    : {len(df_master):,} baris")

# -------------------------------------------------------------
# 2. SAPU BERSIH SELURUH KATA BOCOR DENGAN POLA YANG SAMA
# -------------------------------------------------------------
# Langsung cabut kata 'imposter', 'periksa fakta', dan 'faktanya' di mana pun berada
target_kebocoran = [
    r'\bimposter\b',
    r'\bhasil\s+periksa\s+fakta\b\s*:?',
    r'\bperiksa\s+fakta\b\s*:?',
    r'\bfaktanya\b\s*[:,]?'
]

print("--> Mencabut kata 'imposter', 'periksa fakta', dan 'faktanya'...")
for p in target_kebocoran:
    df_master['text'] = df_master['text'].str.replace(p, '', case=False, regex=True)

# Normalisasi spasi dan tanda titik
df_master['text'] = df_master['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df_master['text'] = df_master['text'].apply(lambda x: x if str(x).endswith(('.', '!', '?')) else str(x) + ".")

# Filter minimal 10 kata agar tidak ada naskah kerdil
df_master['wc'] = df_master['text'].apply(lambda x: len(str(x).split()))
df_master = df_master[df_master['wc'] >= 10].drop(columns=['wc']).copy()

# Penyeimbangan Ulang 50:50 Presisi
min_c = min(df_master['label'].value_counts())
df_h = df_master[df_master['label'] == 1].sample(n=min_c, random_state=42)
df_f = df_master[df_master['label'] == 0].sample(n=min_c, random_state=42)

df_clean_master = pd.concat([df_h, df_f]).sample(frac=1, random_state=42).reset_index(drop=True)
df_clean_master['id'] = [f"POL-{i+1:05d}" for i in range(len(df_clean_master))]

# Simpan kembali master yang steril
df_clean_master.to_csv(input_csv, index=False, encoding='utf-8-sig')

# -------------------------------------------------------------
# 3. VERIFIKASI ULANG KEBOCORAN (PASTI 0)
# -------------------------------------------------------------
sisa_bocor = df_clean_master['text'].str.contains(r'\bimposter\b|\bperiksa\s+fakta\b|\bfaktanya\b', case=False, regex=True).sum()
print(f"Hasil Verifikasi Ulang Kebocoran : {sisa_bocor} baris bocor!")
assert sisa_bocor == 0, f"Masih ada {sisa_bocor} baris bocor!"
print("✅ STATUS: MASTER DATASET RESMI STERIL MUTLAK 100% (0 BOCOR)!")

# -------------------------------------------------------------
# 4. STRATIFIED SPLIT 70% TRAIN, 15% VAL, 15% TEST
# -------------------------------------------------------------
print("\n" + "-" * 75)
print("📦 MEMBAGI MENJADI 3 SUBSET RESMI (70% / 15% / 15%)...")

train_df, temp_df = train_test_split(
    df_clean_master, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_clean_master['label']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_df['label']
)

# Simpan ke folder data/processed/
PROCESSED_DIR = "../data/processed" if os.path.exists("../data/processed") else "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_path = os.path.join(PROCESSED_DIR, "train.csv")
val_path   = os.path.join(PROCESSED_DIR, "val.csv")
test_path  = os.path.join(PROCESSED_DIR, "test.csv")

train_df.to_csv(train_path, index=False, encoding='utf-8-sig')
val_df.to_csv(val_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_path, index=False, encoding='utf-8-sig')

# =============================================================
# 5. LAPORAN HASIL AKHIR DI DISK
# =============================================================
print("=" * 75)
print(f"🏆 BERHASIL! TIGA FILE RESMI TERSIMPAN DI data/processed/ :")
print(f"• train.csv (70%) : {len(train_df):,} baris (Hoaks: {train_df['label'].sum():,} | Fakta: {len(train_df) - train_df['label'].sum():,})")
print(f"• val.csv   (15%) : {len(val_df):,} baris  (Hoaks: {val_df['label'].sum():,} | Fakta: {len(val_df) - val_df['label'].sum():,})")
print(f"• test.csv  (15%) : {len(test_df):,} baris  (Hoaks: {test_df['label'].sum():,} | Fakta: {len(test_df) - test_df['label'].sum():,})")
print(f"TOTAL DATASET    : {len(df_clean_master):,} BARIS (50:50 SEIMBANG)")
print("=" * 75)

🚀 FASE 2: PEMBERSIHAN MUTLAK & STRATIFIED SPLIT 70/15/15
Total Master Dataset Awal    : 14,276 baris
--> Mencabut kata 'imposter', 'periksa fakta', dan 'faktanya'...
Hasil Verifikasi Ulang Kebocoran : 0 baris bocor!
✅ STATUS: MASTER DATASET RESMI STERIL MUTLAK 100% (0 BOCOR)!

---------------------------------------------------------------------------
📦 MEMBAGI MENJADI 3 SUBSET RESMI (70% / 15% / 15%)...
🏆 BERHASIL! TIGA FILE RESMI TERSIMPAN DI data/processed/ :
• train.csv (70%) : 9,993 baris (Hoaks: 4,997 | Fakta: 4,996)
• val.csv   (15%) : 2,141 baris  (Hoaks: 1,070 | Fakta: 1,071)
• test.csv  (15%) : 2,142 baris  (Hoaks: 1,071 | Fakta: 1,071)
TOTAL DATASET    : 14,276 BARIS (50:50 SEIMBANG)
